# Fresh-runtime reproduction: retrained Gaussian + scaled DPAR
This notebook is designed for a **new Colab runtime**. It clones the repository, installs dependencies, runs unit tests, then reproduces the Gaussian denoiser and the calibration/held-out follow-up from scratch. A GPU runtime is strongly recommended.


In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/steering-manifold-repair')
if repo.exists(): subprocess.run(['rm','-rf',str(repo)], check=True)
subprocess.run(['git','clone','https://github.com/Nek1tt/steering-manifold-repair.git',str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'], check=True)
subprocess.run([sys.executable,'-m','pip','install','-e','.'], check=True)
print('cwd:', os.getcwd())


In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available(): print('gpu:', torch.cuda.get_device_name(0))


## Fast code-level reproducibility checks


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_inference_followups.py','tests/test_denoiser.py'], check=True)


## Full fresh-runtime reproduction
This rebuilds the sentiment direction if absent, performs a real-data preflight, caches generic layer-6 activations, retrains the Gaussian denoiser, verifies >50% validation MSE improvement, calibrates beta, and evaluates the frozen held-out prompts/seeds.


In [ ]:
subprocess.run([sys.executable,'scripts/run_retrain_gaussian_followups.py','--config','configs/retrain_gaussian_followups_gpt2.yaml'], check=True)


## Verify the reproduced scientific outputs


In [ ]:
import json, pandas as pd
from pathlib import Path
history = json.loads(Path('results/retrained_denoiser_gaussian_history.json').read_text())
print('final validation improvement:', history[-1]['val_relative_mse_improvement'])
print(Path('results/retrained_inference_followups/SUMMARY.md').read_text())
display(pd.read_csv('results/retrained_inference_followups/heldout_interpolated_frontier.csv'))


Expected reference: final Gaussian validation relative MSE improvement ≈ **0.6781128742**. Small environment-dependent generation differences are possible if upstream library/model implementations change; the repository freezes prompts, seeds, model name, hook, training seed, and evaluation protocol.
